# [문제] 타이타닉 데이터(./titanic/train.csv)를 사용하여 아래 처리 조건을 먼저 수행 한 다음 VIF를 사용하여 다중공선성을 제거한 피처를 학습한 모델을 작성하여 해당 모델의 요약정보(summary())를 출력하는 프로그램을 완성하시오.
(단, 최적의 결과가 나오도록 최선을 다해보시오!!!)


<처리조건>
- target : Survived
- Passengerid, name, Ticket, Cabin 컬름은 삭제
- 누락된 데이터(NaN)가 있는 행은 삭제
- Sex, Emabrked는 레이블 인코딩

In [1]:
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv('./titanic/train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
# 처리조건 1 : target : Survived
import statsmodels.formula.api as smf
target = 'Survived'
features = [col for col in df.columns if col != target]
formula = f"{target} ~ " + " + ".join(features)
formula

'Survived ~ PassengerId + Pclass + Name + Sex + Age + SibSp + Parch + Ticket + Fare + Cabin + Embarked'

In [3]:
# 처리조건 2 : PassengerId, name, Ticket, Cabin 컬럼 삭제

df1 = df.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)
df1.head(10)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
5,0,3,male,NaN,0,0,8.4583,Q
6,0,1,male,54.0,0,0,51.8625,S
7,0,3,male,2.0,3,1,21.0750,S
8,1,3,female,27.0,0,2,11.1333,S
9,1,2,female,14.0,1,0,30.0708,C


In [4]:
# 처리조건 3 : 누락된 데이터(Nan)가있는 행은 삭제
df1 = df1.dropna() # na 값이 있는 행을 drop 시키는 코드
print(df1.isnull().sum())
df1.head(10)

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
6,0,1,male,54.0,0,0,51.8625,S
7,0,3,male,2.0,3,1,21.0750,S
8,1,3,female,27.0,0,2,11.1333,S
9,1,2,female,14.0,1,0,30.0708,C
10,1,3,female,4.0,1,1,16.7000,S


In [5]:
# 처리조건 4 : Sex, Embarked 레이블 인코딩

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df2 = df1.copy()

df2['Sex'] = le.fit_transform(df1['Sex'])
df2['Embarked'] = le.fit_transform(df1['Embarked'])

df2.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,2
1,1,1,0,38.0,1,0,71.2833,0
2,1,3,0,26.0,0,0,7.9250,2
3,1,1,0,35.0,1,0,53.1000,2
4,0,3,1,35.0,0,0,8.0500,2


In [6]:
target = 'Survived'
features = [col for col in df2.columns if col != target]
formula_2 = f"{target} ~ " + " + ".join(features)

from patsy import dmatrices   # 디자인 행렬 함수: 종속변수와 독립변수를 분할 시켜줌
y, X = dmatrices(formula_2, df2, return_type="dataframe")

In [7]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor # VIF 계산 함수

vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])] # 피쳐 단위로 계산해서 리스트를 만들어주고
# 'VIF Factor 의 변수에 저장한다.
vif["features"] = X.columns
vif

,VIF Factor,features
0,32.368412,Intercept
1,1.749120,Pclass
2,1.119651,Sex
3,1.310796,Age
4,1.276469,SibSp
5,1.288619,Parch
6,1.641348,Fare
7,1.114880,Embarked


In [8]:
from patsy import dmatrices
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

def get_vif(formula_2, df):
    y, X = dmatrices(formula_2, df2, return_type="dataframe")
    vif = pd.DataFrame()
    vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])] 
    vif["features"] = X.columns
    return vif

In [9]:
target = 'Survived'
features = [col for col in df2.columns if col != target]
formula_2 = f"{target} ~ " + " + ".join(features)
print(formula_2)
get_vif(formula_2, df2)

Survived ~ Pclass + Sex + Age + SibSp + Parch + Fare + Embarked


,VIF Factor,features
0,32.368412,Intercept
1,1.749120,Pclass
2,1.119651,Sex
3,1.310796,Age
4,1.276469,SibSp
5,1.288619,Parch
6,1.641348,Fare
7,1.114880,Embarked


In [10]:
# intercept의 vif가 32.42로 10을 넘으니 제거
formula_2 = "Survived ~ Pclass + Sex + Age + SibSp + Parch + Fare + Embarked - 1"
print(formula_2)
get_vif(formula_2, df2)

Survived ~ Pclass + Sex + Age + SibSp + Parch + Fare + Embarked - 1


,VIF Factor,features
0,5.990097,Pclass
1,3.035750,Sex
2,4.169982,Age
3,1.638657,SibSp
4,1.618635,Parch
5,1.659422,Fare
6,5.282948,Embarked


In [11]:
# pclass 제거
formula_2 = "Survived ~ Sex + Age + SibSp + Parch + Fare + Embarked - 1"
print(formula_2)
get_vif(formula_2, df2)

Survived ~ Sex + Age + SibSp + Parch + Fare + Embarked - 1


,VIF Factor,features
0,2.680324,Sex
1,3.928776,Age
2,1.583900,SibSp
3,1.560488,Parch
4,1.593394,Fare
5,3.818932,Embarked


In [12]:
# Sibsp 제거
formula_2 = "Survived ~ Sex + Age + Parch + Fare + Embarked - 1"
print(formula_2)
get_vif(formula_2, df2)

Survived ~ Sex + Age + Parch + Fare + Embarked - 1


,VIF Factor,features
0,2.655172,Sex
1,3.799646,Age
2,1.355155,Parch
3,1.536595,Fare
4,3.612633,Embarked


In [13]:
# Parch 제거
formula_2 = "Survived ~ Sex + Age + Fare + Embarked - 1"
print(formula_2)
get_vif(formula_2, df2)

Survived ~ Sex + Age + Fare + Embarked - 1


,VIF Factor,features
0,2.608970,Sex
1,3.782898,Age
2,1.393202,Fare
3,3.257920,Embarked


In [14]:
# 처리조건 5 : OLS모델을 사용해서 요약정보 출력
# 피쳐 제거 후, 학습한 모델의 요약정보 출력
import statsmodels.formula.api as nmf

model_titanic_final = smf.ols(formula=formula_2, data=df2).fit()
print(model_titanic_final.summary())

                                 OLS Regression Results                                
Dep. Variable:               Survived   R-squared (uncentered):                   0.464
Model:                            OLS   Adj. R-squared (uncentered):              0.461
Method:                 Least Squares   F-statistic:                              153.4
Date:                Tue, 28 Apr 2026   Prob (F-statistic):                    1.87e-94
Time:                        17:44:06   Log-Likelihood:                         -465.89
No. Observations:                 712   AIC:                                      939.8
Df Residuals:                     708   BIC:                                      958.1
Df Model:                           4                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------